# 1kg_eur — rare-variant PCA, odd vs even chromosome replication

Does each rare PC describe genome-wide structure, or one locus?

Refit each PC set on **odd chromosomes only** and on **even chromosomes only**,
then correlate the two across individuals. A PC carried by real genome-wide
structure is recoverable from either half of the genome, so |r| between the odd
fit and the even fit is high. A PC carried by a single locus, an LD block, or one
family's shared haplotype lives on *one* chromosome — so it appears in one half
and not the other, and the correlation collapses.

This is the diagnostic from `s41467-026-73776-9` (they generate odd- and
even-chromosome rare PCs independently for exactly this reason). It is a stronger
test than reading the loadings plot, because it needs no judgement about what
counts as a spike.

**Reuses everything from `rare_pca_unrelated.ipynb`** — same panel
(`~/scratch_1kg_eur_rare/qc/rare_qc`), same per-arm `.prune.in`, same
`unrel/unrelated_keep_ids.txt`. Only `--chr` is added. Separate notebook rather
than more cells in that one, since its four fits have already run.

**8 fits:** 2 arms x {odd, even} x {all, unrel}. Running odd/even on both sample
sets answers a second question for free: whether relatedness is what breaks
replication. If a PC replicates in `unrel` but not in `all`, the thing carrying
it was a family.

Expect the two arms to behave differently and read them differently. From the
completed run:

| arm | variants | per half |
|---|---|---|
| `hm3_rare_all` | 5,484 | ~2,700 |
| `nonhm3_rare_all` | 174,682 | ~87,000 |

At ~2,700 variants for 223K samples the HM3 arm is deep in p << n, where even
genuine structure replicates poorly simply from sampling noise. Poor replication
there is weak evidence of an artefact. In the non-HM3 arm it means something.

## Config

In [ ]:
import os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SAMPLE_SET = "1kg_eur"
WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
OUT_GS = f"{WS_GS}/{SAMPLE_SET}/01_ancestry/rare_pca_unrelated"

RARE  = os.path.expanduser(f"~/scratch_{SAMPLE_SET}_rare")
PANEL = f"{RARE}/qc/rare_qc"
ARMS  = ["hm3_rare_all", "nonhm3_rare_all"]

# written by rare_pca_unrelated.ipynb
PREV       = f"{RARE}/unrel"
KEEP_ALL   = f"{RARE}/round2_keep.txt"
UNREL_KEEP = f"{PREV}/unrelated_keep_ids.txt"

OUT = f"{RARE}/oddeven"
os.makedirs(OUT, exist_ok=True)

N_PCS_FIT = 10   # matches the fits being diagnosed
SPLITS = {
    "odd":  "1,3,5,7,9,11,13,15,17,19,21",
    "even": "2,4,6,8,10,12,14,16,18,20,22",
}
KEEPS = {"all": KEEP_ALL, "unrel": UNREL_KEEP}

for p in (f"{PANEL}.pgen", KEEP_ALL, UNREL_KEEP,
          *(f"{RARE}/{a}.prune.in" for a in ARMS)):
    print(f"  {'ok  ' if os.path.isfile(p) else 'MISS'} {p}")
print(f"\nout: {OUT}")

## How the pruned variants split across chromosomes

Check before fitting. The split is only meaningful if both halves carry a
comparable number of variants — and if one arm's variants are concentrated on a
few chromosomes, that is itself worth seeing.

In [ ]:
chrom_counts = {}
for arm in ARMS:
    ids = pd.read_csv(f"{RARE}/{arm}.prune.in", header=None, names=["ID"])
    chrom = (ids["ID"].str.split(":", n=1, expand=True)[0]
             .str.replace("^chr", "", regex=True))
    chrom_counts[arm] = chrom.value_counts()
    odd  = [str(c) for c in range(1, 23, 2)]
    even = [str(c) for c in range(2, 23, 2)]
    n_o = int(chrom_counts[arm].reindex(odd).fillna(0).sum())
    n_e = int(chrom_counts[arm].reindex(even).fillna(0).sum())
    print(f"{arm:16s} total {len(ids):>7,}   odd {n_o:>7,}   even {n_e:>7,}")

order = [str(c) for c in range(1, 23)]
fig, axes = plt.subplots(len(ARMS), 1, figsize=(11, 3 * len(ARMS)), squeeze=False)
for ax, arm in zip(axes[:, 0], ARMS):
    v = chrom_counts[arm].reindex(order).fillna(0)
    ax.bar(range(1, 23), v.to_numpy(),
           color=["tab:blue" if c % 2 else "tab:orange" for c in range(1, 23)])
    ax.set_xticks(range(1, 23)); ax.set_ylabel("variants"); ax.set_title(arm)
    ax.set_xlabel("chromosome (blue = odd, orange = even)")
plt.tight_layout(); plt.savefig(f"{OUT}/variants_per_chrom.png", dpi=150); plt.show()

## Fit — 8 runs

Each is `--extract` the arm's pruned list, `--keep` the sample set, `--chr` the
half. Skips any fit whose `.eigenvec` already exists, so this is safe to re-run.

The full-genome fits already exist in `unrel/`, so they are not recomputed here.

In [ ]:
for arm in ARMS:
    prune_in = f"{RARE}/{arm}.prune.in"
    for fit, keep in KEEPS.items():
        for split, chrs in SPLITS.items():
            out = f"{OUT}/{arm}_{fit}_{split}"
            if os.path.isfile(f"{out}.eigenvec"):
                print(f"exists, skipping: {os.path.basename(out)}")
                continue
            print(f"=== {arm} / {fit} / {split}")
            subprocess.run(["bash", "-c", f"""
set -eo pipefail
export PATH="$HOME/bin:$PATH"
plink2 --pfile "{PANEL}" --nonfounders \
  --keep "{keep}" --extract "{prune_in}" --chr {chrs} \
  --freq counts --pca approx {N_PCS_FIT} allele-wts \
  --threads $(nproc) --out "{out}"
"""], check=True)

## Load

In [ ]:
PC_COLS = [f"PC{k}" for k in range(1, N_PCS_FIT + 1)]

def load(stem):
    if not os.path.isfile(f"{stem}.eigenvec"):
        return None, None
    d = pd.read_csv(f"{stem}.eigenvec", sep=r"\s+")
    idc = "#IID" if "#IID" in d.columns else "IID"
    d = d.rename(columns={idc: "person_id"})
    d["person_id"] = d["person_id"].astype(str)
    ev = np.loadtxt(f"{stem}.eigenval")
    return d.set_index("person_id")[PC_COLS], ev / ev.sum() * 100

half, half_pct, full, full_pct = {}, {}, {}, {}
for arm in ARMS:
    for fit in KEEPS:
        f, fp = load(f"{PREV}/{arm}_{fit}")          # full-genome fit, previous notebook
        if f is not None:
            full[(arm, fit)], full_pct[(arm, fit)] = f, fp
        for split in SPLITS:
            h, hp = load(f"{OUT}/{arm}_{fit}_{split}")
            if h is not None:
                half[(arm, fit, split)], half_pct[(arm, fit, split)] = h, hp

for k in sorted(half):
    print(f"{k[0]:16s} {k[1]:6s} {k[2]:5s} n={len(half[k]):>7,}  "
          + "  ".join(f"PC{i+1} {half_pct[k][i]:.2f}%" for i in range(3)))

## Odd vs even correlation

PCs are identified only up to sign, and within a near-degenerate eigenvalue
subspace only up to rotation — so the honest statistic is `max |r|` of each odd
PC against *all* even PCs, not the diagonal. Both are reported: a high `max_abs_r`
with a low `diag_abs_r` and a `best_match` off the diagonal means the axis
replicates but the PCs come out in a different order, which is not a problem.

In [ ]:
def odd_even(arm, fit):
    o, e = half.get((arm, fit, "odd")), half.get((arm, fit, "even"))
    if o is None or e is None:
        return None
    ids = o.index.intersection(e.index)
    R = np.corrcoef(o.loc[ids].to_numpy().T,
                    e.loc[ids].to_numpy().T)[:N_PCS_FIT, N_PCS_FIT:]
    return pd.DataFrame({
        "PC": range(1, N_PCS_FIT + 1),
        "diag_abs_r": np.abs(np.diag(R)),
        "max_abs_r": np.abs(R).max(axis=1),
        "best_match": np.abs(R).argmax(axis=1) + 1,
        "pct_var_full": (full_pct[(arm, fit)][:N_PCS_FIT]
                         if (arm, fit) in full_pct else np.nan),
        "n": len(ids),
    }).set_index("PC")

tabs = {}
for arm in ARMS:
    for fit in KEEPS:
        t = odd_even(arm, fit)
        if t is None:
            continue
        tabs[(arm, fit)] = t
        print(f"\n── {arm} / {fit}  (n={t['n'].iloc[0]:,})")
        print(t.drop(columns="n").round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, len(ARMS), figsize=(7 * len(ARMS), 4.5), squeeze=False)
for ax, arm in zip(axes[0], ARMS):
    for fit, style in (("all", "-o"), ("unrel", "--s")):
        if (arm, fit) in tabs:
            ax.plot(tabs[(arm, fit)].index, tabs[(arm, fit)]["max_abs_r"],
                    style, ms=5, label=fit)
    ax.axhline(0.9, color="0.7", ls=":", lw=1)
    ax.set_ylim(0, 1.02); ax.set_xticks(range(1, N_PCS_FIT + 1))
    ax.set_xlabel("PC"); ax.set_ylabel("max |r| (odd vs even)")
    ax.legend(); ax.set_title(arm)
plt.suptitle("Odd/even replication — below the dotted line is not genome-wide structure")
plt.tight_layout(); plt.savefig(f"{OUT}/odd_even_replication.png", dpi=150); plt.show()

## Does the half-genome PC agree with the full-genome PC?

A second framing of the same question, and a check on the one above. Each
half-genome fit is correlated against the full-genome fit it came from. A PC
driven by one locus on, say, an odd chromosome will match the odd fit well and
the even fit hardly at all — an asymmetry the odd-vs-even number alone hides.

In [ ]:
rows = []
for (arm, fit, split), h in half.items():
    if (arm, fit) not in full:
        continue
    f = full[(arm, fit)]
    ids = h.index.intersection(f.index)
    R = np.corrcoef(h.loc[ids].to_numpy().T,
                    f.loc[ids].to_numpy().T)[:N_PCS_FIT, N_PCS_FIT:]
    for k in range(N_PCS_FIT):
        rows.append({"arm": arm, "fit": fit, "split": split, "PC": k + 1,
                     "max_abs_r_vs_full": np.abs(R[k]).max()})
vs_full = (pd.DataFrame(rows)
           .pivot(index=["arm", "fit", "PC"], columns="split",
                  values="max_abs_r_vs_full"))
vs_full["asymmetry"] = (vs_full["odd"] - vs_full["even"]).abs()
print("|r| of each half-genome PC against the full-genome fit:")
print(vs_full.round(3).to_string())
print("\nlarge asymmetry = the PC lives in one half of the genome:")
print(vs_full[vs_full["asymmetry"] > 0.3].round(3).to_string() or "  none")

## Summary and upload

In [ ]:
parts = []
for (arm, fit), t in tabs.items():
    x = t.drop(columns="n").reset_index()
    x["arm"], x["fit"] = arm, fit
    parts.append(x)
summary = pd.concat(parts, ignore_index=True)
SUMMARY = f"{OUT}/odd_even_summary.tsv"
summary.to_csv(SUMMARY, sep="\t", index=False, float_format="%.5f")
print(summary.round(3).to_string(index=False))

subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{SUMMARY}" "{OUT_GS}/odd_even_summary.tsv"
gcloud storage cp "{OUT}/"*.png "{OUT_GS}/figures/"
gcloud storage cp "{OUT}/"*.eigenval "{OUT_GS}/oddeven/"
"""], check=True)
print(f"\nuploaded to {OUT_GS}")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/rare_pca_oddeven.ipynb')
_gs = f'{OUT_GS}/notebooks/rare_pca_oddeven.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')